# Correcting Fixation Drift



The pymovements library provides a collection of built-in fixation drift correction algorithms to be used on gaze data.

These algorithms use various methods to automatically correct fixations that are deemed inaccurate due to measurement issues such as noise, slope, or shift.

In this tutorial you'll learn how to:

- load a reading dataset and prepare fixations
- map fixations to the AOIs
- apply fixation drift correction algorithms, along with the WoC ensemble
- pick a specific algorithm for drift correction
- visualize the results before and after correction

All examples use the GGTG Dataset that comes with pymovements

## Loading Data

We begin by loading the GGTG dataset, which ships with pymovements. To keep the tutorial fast and focused, we restrict loading to a single subject (P01). The GGTG dataset is a good fit for this tutorial because it includes precomputed fixations as well as text stimulus AOI files, so we don't need to run event detection ourselves.

In [21]:
from pathlib import Path

import polars as pl

import pymovements
from pymovements.events.correction.fixation_correction import \
    correct_fixation_locations

dataset = pymovements.Dataset("GGTG", path="data/GGTG")

dataset.download()

dataset.load(
    subset={"subject_id": "P01"},
)

INFO:pymovements.dataset.dataset:
        You are downloading the Gaze-Guided Text Generation. Please be aware that pymovements does not
        host or distribute any dataset resources and only provides a convenient interface to
        download the public dataset resources that were published by their respective authors.

        Please cite the referenced publication if you intend to use the dataset in your research.
        


Verifying existing file: data\GGTG\downloads\samples.zip
Using existing verified file: data\GGTG\downloads\samples.zip
Verifying existing file: data\GGTG\downloads\fixations.zip
Using existing verified file: data\GGTG\downloads\fixations.zip
Verifying existing file: data\GGTG\downloads\measures.zip
Using existing verified file: data\GGTG\downloads\measures.zip
Verifying existing file: data\GGTG\downloads\aoi-csvs.zip
Using existing verified file: data\GGTG\downloads\aoi-csvs.zip
Extracting samples.zip to data\GGTG\raw


Extracting archive: 100%|██████████| 24/24 [00:01<00:00, 15.68file/s]


Extracting fixations.zip to data\GGTG\precomputed_events


Extracting archive: 100%|██████████| 24/24 [00:00<00:00, 864.95file/s]


...

Extracting archive: 100%|██████████| 24/24 [00:00<00:00, 945.95file/s]


Extracting aoi-csvs.zip to data\GGTG\stimuli


Extracting archive: 100%|██████████| 316/316 [00:00<00:00, 2403.35file/s]


Loading gaze files:   0%|          | 0/1 [00:00<?, ?file/s]

D:\dionigi\Documents\Python scripts\pyMovementLab\pymovements\src\pymovements\dataset\dataset.py:588: ExperimentalWarning: Stimulus support is experimental. Names and behavior may change without being considered a breaking change. Please set the used pymovements version explicitly to prevent unexptected changes. The used pymovements version is v0.26.0+post19.8d7fea44.
  warn(


## Extracting fixations from data 

Our dataset already has text stimuli, which contain the words and their locations, of the stimulus the fixations were measured on, but we first need to extract the fixations so we can then map them to their respective text stimuli.

The GGTG dataset stores precomputed fixations in a separate file. We select fixations belonging to a single stimulus (goldfish-pos.text.0) and keep only the columns we need: the stimulus identifier, onset, offset, duration, and the location_x / location_y coordinates of each fixation. Finally, we add a name column with the value "fixation", which pymovements uses to identify the event type.

In [22]:
stimulus_name = "goldfish-pos.text.0"

fixation_path = (
    dataset.paths.precomputed_events
    / dataset.fileinfo["precomputed_events"]["filepath"][0]
)

fixations = (
    pl.read_csv(fixation_path)
    .filter(pl.col("stimulus") == stimulus_name)
    .select(
        "stimulus",
        "onset",
        "offset",
        "duration",
        "location_x",
        "location_y",
    )
    .with_columns(pl.lit("fixation").alias("name"))
)
events_raw = pymovements.Events(fixations)

## Extracting text stimulus and creating a TextStimulus object

To provide a visualization for us to plot our fixations onto, we must first prepare a TextStimulus object.

A TextStimulus holds the AOIs (here, individual words) of the stimulus, together with their bounding boxes on screen. We look up the AOI file that corresponds to our chosen stimulus and word-level units, read it as a Polars DataFrame, and construct the TextStimulus by telling it which columns hold the word content and the bounding box coordinates (left/top/right/bottom).

In [23]:
aoi_path = (
    dataset.paths.stimuli
    / dataset.fileinfo["textstimulus"]
    .filter(
        (pl.col("stimulus") == stimulus_name)
        & (pl.col("unit") == "word")
    )["filepath"][0]
)

aois = pl.read_csv(aoi_path)

text_stimulus = pymovements.stimulus.TextStimulus(
    aois=aois,
    aoi_column="content",
    start_x_column="left",
    start_y_column="top",
    end_x_column="right",
    end_y_column="bottom",
)

## Mapping fixations to AOIs

Before correcting anything, we should quantify how well the raw fixations land on their intended words. This is exactly what TextStimulus.get_aoi() is for: given a fixation's eye-coordinate row, it returns the AOI (here, the word) whose bounding box contains that point.

We'll call get_aoi() once per fixation, collect the matched word into a new aoi_content column, and then count how many fixations are unassigned.

In [ ]:

    
matched = []
    
for row in events_raw.frame.iter_rows(named=True):
    aoi = text_stimulus.get_aoi(
        row=row,
        x_eye="location_x",
        y_eye="location_y")
           
    matched.append(aoi["content"][0] if aoi.height and aoi["content"][0] is not None else None)

events_raw.frame = events_raw.frame.with_columns(pl.Series("aoi_content", matched))


n_total = events_raw.frame.height

n_unmatched =  events_raw.frame["aoi_content"].is_null().sum()

print(f"Raw fixations: {n_total} total, {n_unmatched} unmatched " f"({100 * n_unmatched / n_total:.1f}%)")

Raw fixations: 94 total, 3 unmatched (3.2%)


## Applying fixation correction algorithms

To apply our fixation correction algorithms, we first create Events objects from our extracted fixations, after which we can apply correct_fixations() to them in the following forms with various algorithms depending on the user's choice.

Fixation drift correction addresses a common measurement problem in eye tracking: even when a participant is actually looking at a word, the recorded fixation location may be slightly offset (drifted), for example, a few pixels above or below the word. Because downstream analyses such as AOI mapping and reading measures are sensitive to these offsets, correcting fixations before analysis can substantially improve data quality.

Events.correct_fixations() takes a TextStimulus and an algorithm argument. We demonstrate three options:

"wisdom_of_the_crowd" (WoC): an ensemble method that combines multiple correction algorithms and is a sensible default.

"warp": a single named algorithm based on a warp/registration approach.

"chain": another single named algorithm that chains neighboring fixations together.

We create a separate Events object for each algorithm.

In [ ]:

events_woc = pymovements.Events(fixations)
events_warp = pymovements.Events(fixations)
events_chain = pymovements.Events(fixations)

events_woc.correct_fixations(
    text_stimulus,
    algorithm="wisdom_of_the_crowd",
)

events_warp.correct_fixations(
    text_stimulus,
    algorithm="warp",
)

events_chain.correct_fixations(
    text_stimulus,
    algorithm="chain",
)

To pick a specific algorithm rather than the WoC ensemble, simply pass its name to the algorithm parameter, as shown above with "warp" and "chain". The WoC ensemble is a good default when you don't have a strong reason to prefer one algorithm; named algorithms are useful when you want a deterministic, interpretable correction or want to compare methods.



## Plotting pre and post-correction scanpaths

Before plotting our scanpaths, we concatenate our location columns as scanpathplot() expects a single column of [x, y] coordinates. After this, we add the events to a Gaze object and pass it to scanpathplot() for our final plots.

We first plot the raw (uncorrected) fixations on top of the text stimulus so we have a baseline to compare against.

In [ ]:
events_raw = pymovements.Events(
    events_raw.frame.with_columns(
        pl.concat_list(["location_x", "location_y"]).alias("location"),
    ),
)

gaze_for_plot = pymovements.Gaze(events=events_raw)

fig, ax = text_stimulus.plot()

fig, ax = pymovements.plotting.scanpathplot(
    gaze=gaze_for_plot,
    position_column="location",
    event_name="fixation",
    ax=ax,
    color="red",
    alpha=0.7,
    add_arrows=True,
    title="Raw fixation scanpath",
)

The raw scanpath shows fixations drifting slightly above or below their target words. The cell below repeats the same plotting procedure, but using the WoC-corrected events. Comparing the two figures side by side shows how the correction pulls drifted fixations back onto their intended words, which in turn improves AOI assignment and any reading measures computed downstream. You can use this before/after comparison to decide whether correction is helping for your data: if fixations were already well-centered on words, correction should change little; if they were visibly drifted, the corrected scanpath should sit more cleanly within word boundaries.

In [ ]:
events_woc = pymovements.Events(
    events_woc.frame.with_columns(
        pl.concat_list(["location_x", "location_y"]).alias("location"),
    ),
)

gaze_for_plot = pymovements.Gaze(events=events_woc)

fig, ax = text_stimulus.plot()

fig, ax = pymovements.plotting.scanpathplot(
    gaze=gaze_for_plot,
    position_column="location",
    event_name="fixation",
    ax=ax,
    color="red",
    alpha=0.7,
    add_arrows=True,
    title="Corrected fixation scanpath",
)